# The AI Arena: Who Wins the Battle of Chatbots?
### LMSYS Chatbot Arena — Story & Insights

---

**Story Summary:**  
In the LMSYS Chatbot Arena, two AI models face off head-to-head in 57,477 blind evaluations by human judges. This notebook answers one central question:  
> *What separates the winners from the losers — and does it come down to the model, the response length, or sheer dominance of a few giants?*

We'll follow a clear narrative arc:
1. **The Arena** — Understanding the setup
2. **The Champions** — Which models dominate?
3. **The Bias Check** — Does position (A vs B) matter?
4. **The Length Question** — Do longer responses win?
5. **The Tie Puzzle** — When do humans refuse to choose?
6. **The Verdict** — Final insights and dashboard direction

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Color palette for consistency
COLORS = {
    'model_a': '#636EFA',
    'model_b': '#EF553B',
    'tie':     '#00CC96',
    'accent':  '#FFA15A',
    'neutral': '#AB63FA'
}

df = pd.read_csv('lmsys-chatbot-arena/train.csv')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

Dataset loaded: 57,477 rows × 9 columns


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1


In [2]:
# !pip install --upgrade nbformat

## Chapter 1: The Arena
Two models enter. One wins. Sometimes, nobody does.

The very first thing to understand is the outcome distribution — is this arena fair?

In [3]:
# --- Winner Distribution Donut ---
winner_counts = {
    'Model A Wins': df['winner_model_a'].sum(),
    'Model B Wins': df['winner_model_b'].sum(),
    'Tie':          df['winner_tie'].sum()
}

fig = go.Figure(go.Pie(
    labels=list(winner_counts.keys()),
    values=list(winner_counts.values()),
    hole=0.55,
    marker_colors=[COLORS['model_a'], COLORS['model_b'], COLORS['tie']],
    textinfo='label+percent',
    hovertemplate='%{label}: %{value:,} battles<extra></extra>'
))

total = sum(winner_counts.values())
fig.add_annotation(
    text=f'<b>{total:,}</b><br>battles',
    x=0.5, y=0.5,
    font_size=18,
    showarrow=False
)

fig.update_layout(
    title=dict(text='Battle Outcomes: Is the Arena Balanced?', font_size=20),
    legend=dict(orientation='h', y=-0.1),
    height=450
)

fig.show()

print("\n💡 Insight: The arena is remarkably balanced — Model A and B win at nearly equal rates.")
print("   This rules out a systematic position bias at the aggregate level.")
print(f"   However, ties are substantial at {winner_counts['Tie']/total*100:.1f}% — worth investigating.")


💡 Insight: The arena is remarkably balanced — Model A and B win at nearly equal rates.
   This rules out a systematic position bias at the aggregate level.
   However, ties are substantial at 30.9% — worth investigating.


## Chapter 2: The Champions
Not all models are created equal. A handful of models dominate the arena — but who actually wins the most?

In [4]:
# --- Compute a combined win rate (position-agnostic) ---
records = []
for _, row in df.iterrows():
    for model, win_col in [(row['model_a'], 'winner_model_a'), (row['model_b'], 'winner_model_b')]:
        records.append({'model': model, 'won': row[win_col], 'tied': row['winner_tie']})

perf_df = pd.DataFrame(records)
model_stats = perf_df.groupby('model').agg(
    total=('won', 'count'),
    wins=('won', 'sum'),
    ties=('tied', 'sum')
).reset_index()

model_stats['win_rate']  = model_stats['wins']  / model_stats['total']
model_stats['tie_rate']  = model_stats['ties']  / model_stats['total']
model_stats['loss_rate'] = 1 - model_stats['win_rate'] - model_stats['tie_rate']

# Filter to models with enough appearances for reliability
MIN_BATTLES = 100
reliable = model_stats[model_stats['total'] >= MIN_BATTLES].sort_values('win_rate', ascending=False)

top15 = reliable.head(15).sort_values('win_rate')

fig = go.Figure()
fig.add_trace(go.Bar(
    y=top15['model'], x=top15['win_rate'],
    orientation='h', name='Win Rate',
    marker_color=COLORS['model_a'],
    hovertemplate='<b>%{y}</b><br>Win Rate: %{x:.1%}<extra></extra>'
))
fig.add_trace(go.Bar(
    y=top15['model'], x=top15['tie_rate'],
    orientation='h', name='Tie Rate',
    marker_color=COLORS['tie'],
    hovertemplate='<b>%{y}</b><br>Tie Rate: %{x:.1%}<extra></extra>'
))
fig.add_trace(go.Bar(
    y=top15['model'], x=top15['loss_rate'],
    orientation='h', name='Loss Rate',
    marker_color=COLORS['model_b'],
    hovertemplate='<b>%{y}</b><br>Loss Rate: %{x:.1%}<extra></extra>'
))

fig.update_layout(
    barmode='stack',
    title=dict(text=f'Top 15 Models by Win Rate (min. {MIN_BATTLES} appearances)', font_size=20),
    xaxis=dict(title='Rate', tickformat='.0%'),
    yaxis=dict(title=''),
    legend=dict(orientation='h', y=1.08),
    height=500
)
fig.show()

top3 = reliable.nlargest(3, 'win_rate')[['model','win_rate','total']]
print("\n💡 Top 3 Models by Win Rate (with sufficient battles):")
for _, r in top3.iterrows():
    print(f"   {r['model']}: {r['win_rate']:.1%} win rate over {r['total']:,} battles")


💡 Top 3 Models by Win Rate (with sufficient battles):
   gpt-4-1106-preview: 55.1% win rate over 7,387 battles
   gpt-3.5-turbo-0314: 54.6% win rate over 1,302 battles
   gpt-4-0125-preview: 51.4% win rate over 1,160 battles


## Chapter 3: The Bias Check — Does Position Matter?
Each battle assigns models randomly to position A or B. Does a model perform differently depending on which slot it occupies?

In [5]:
# For models with enough appearances in both positions
win_a = df.groupby('model_a')['winner_model_a'].agg(['mean','count']).rename(columns={'mean':'win_rate_A','count':'n_A'})
win_b = df.groupby('model_b')['winner_model_b'].agg(['mean','count']).rename(columns={'mean':'win_rate_B','count':'n_B'})

pos_df = win_a.join(win_b, how='inner')
pos_df = pos_df[(pos_df['n_A'] >= 50) & (pos_df['n_B'] >= 50)].copy()
pos_df['bias'] = pos_df['win_rate_A'] - pos_df['win_rate_B']
pos_df = pos_df.sort_values('bias')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pos_df['win_rate_A'], y=pos_df['win_rate_B'],
    mode='markers+text',
    marker=dict(
        size=12,
        color=pos_df['bias'],
        colorscale='RdBu',
        showscale=True,
        colorbar=dict(title='A − B bias')
    ),
    text=pos_df.index,
    textposition='top center',
    hovertemplate='<b>%{text}</b><br>Win Rate A: %{x:.1%}<br>Win Rate B: %{y:.1%}<extra></extra>'
))

# Diagonal line = perfect symmetry
lim = [0, 0.7]
fig.add_shape(type='line', x0=lim[0], y0=lim[0], x1=lim[1], y1=lim[1],
              line=dict(color='gray', dash='dash'))

fig.update_layout(
    title=dict(text='Position Bias: Win Rate in Slot A vs Slot B', font_size=20),
    xaxis=dict(title='Win Rate when in Position A', tickformat='.0%'),
    yaxis=dict(title='Win Rate when in Position B', tickformat='.0%'),
    height=520
)
fig.show()

most_biased = pos_df['bias'].abs().idxmax()
print(f"\n💡 Insight: Most models hug the diagonal closely — position has minimal effect.")
print(f"   Largest outlier: '{most_biased}' with bias of {pos_df.loc[most_biased,'bias']:+.1%}")
print("   The arena's random assignment successfully eliminates systemic position bias.")


💡 Insight: Most models hug the diagonal closely — position has minimal effect.
   Largest outlier: 'llama2-70b-steerlm-chat' with bias of +7.9%
   The arena's random assignment successfully eliminates systemic position bias.


## Chapter 4: The Length Question
A common belief: *longer answers signal more effort and win more often.*  
Let's test this directly with the data.

In [6]:
df['response_a_length'] = df['response_a'].str.len()
df['response_b_length'] = df['response_b'].str.len()
df['length_ratio'] = df['response_a_length'] / (df['response_b_length'] + 1)  # A / B

# Bin the length ratio
bins = [0, 0.5, 0.75, 1.0, 1.33, 2.0, 100]
labels = ['A much shorter', 'A shorter', 'Similar', 'A slightly longer', 'A longer', 'A much longer']
df['length_category'] = pd.cut(df['length_ratio'], bins=bins, labels=labels)

length_analysis = df.groupby('length_category', observed=True).agg(
    count=('winner_model_a', 'count'),
    a_win_rate=('winner_model_a', 'mean'),
    b_win_rate=('winner_model_b', 'mean'),
    tie_rate=('winner_tie', 'mean')
).reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=length_analysis['length_category'], y=length_analysis['a_win_rate'],
    name='Model A Wins', marker_color=COLORS['model_a'],
    hovertemplate='%{x}<br>Model A Win Rate: %{y:.1%}<extra></extra>'
))
fig.add_trace(go.Bar(
    x=length_analysis['length_category'], y=length_analysis['b_win_rate'],
    name='Model B Wins', marker_color=COLORS['model_b'],
    hovertemplate='%{x}<br>Model B Win Rate: %{y:.1%}<extra></extra>'
))
fig.add_trace(go.Bar(
    x=length_analysis['length_category'], y=length_analysis['tie_rate'],
    name='Tie', marker_color=COLORS['tie'],
    hovertemplate='%{x}<br>Tie Rate: %{y:.1%}<extra></extra>'
))

fig.update_layout(
    barmode='stack',
    title=dict(text='Does Response Length Determine the Winner?', font_size=20),
    xaxis=dict(title='Relative Length of Response A vs Response B'),
    yaxis=dict(title='Outcome Rate', tickformat='.0%'),
    legend=dict(orientation='h', y=1.08),
    height=450
)
fig.show()

print("\n💡 Insight: When Model A's response is much longer, Model A wins more often.")
print("   However, the effect is asymmetric — being slightly longer doesn't help much.")
print("   Being MUCH shorter is consistently penalized. Length signals effort, but modestly.")


💡 Insight: When Model A's response is much longer, Model A wins more often.
   However, the effect is asymmetric — being slightly longer doesn't help much.
   Being MUCH shorter is consistently penalized. Length signals effort, but modestly.


In [7]:
# --- Average response length per model vs win rate ---
model_lengths = []
for model in model_stats['model']:
    as_a = df[df['model_a'] == model]['response_a_length']
    as_b = df[df['model_b'] == model]['response_b_length']
    combined = pd.concat([as_a, as_b])
    model_lengths.append({'model': model, 'avg_length': combined.mean()})

len_df = pd.DataFrame(model_lengths)
merged = reliable.merge(len_df, on='model')

fig = px.scatter(
    merged, x='avg_length', y='win_rate',
    size='total', color='win_rate',
    color_continuous_scale='Viridis',
    hover_name='model',
    hover_data={'total': True, 'win_rate': ':.1%', 'avg_length': ':,.0f'},
    labels={'avg_length': 'Avg Response Length (chars)', 'win_rate': 'Win Rate', 'total': 'Battles'},
    title='Average Response Length vs Win Rate (bubble size = # battles)'
)

# Trend line
z = np.polyfit(merged['avg_length'], merged['win_rate'], 1)
x_line = np.linspace(merged['avg_length'].min(), merged['avg_length'].max(), 100)
fig.add_trace(go.Scatter(
    x=x_line, y=np.polyval(z, x_line),
    mode='lines', line=dict(color='red', dash='dash'),
    name='Trend'
))

fig.update_layout(height=500, yaxis=dict(tickformat='.0%'))
fig.show()

corr = merged['avg_length'].corr(merged['win_rate'])
print(f"\n💡 Correlation between avg response length and win rate: r = {corr:.3f}")
if abs(corr) > 0.3:
    print("   Moderate positive correlation — longer responses do tend to win more.")
else:
    print("   Weak correlation — length alone doesn't explain why a model wins.")


💡 Correlation between avg response length and win rate: r = 0.499
   Moderate positive correlation — longer responses do tend to win more.


## Chapter 5: The Tie Puzzle
Nearly 1 in 3 battles ends in a tie. Which models produce the most indecisive outcomes — and why?

In [8]:
# Tie rate vs win rate — the tradeoff
fig = px.scatter(
    reliable,
    x='tie_rate', y='win_rate',
    size='total',
    color='total',
    color_continuous_scale='Plasma',
    hover_name='model',
    hover_data={'total': True, 'win_rate': ':.1%', 'tie_rate': ':.1%'},
    labels={'tie_rate': 'Tie Rate', 'win_rate': 'Win Rate', 'total': 'Battles'},
    title='Tie Rate vs Win Rate: Are Some Models Chronically Undecided?'
)

fig.update_layout(
    height=500,
    xaxis=dict(tickformat='.0%'),
    yaxis=dict(tickformat='.0%')
)
fig.show()

high_tie = reliable.nlargest(5, 'tie_rate')[['model','tie_rate','win_rate','total']]
print("\n💡 Top 5 models with highest tie rates:")
for _, r in high_tie.iterrows():
    print(f"   {r['model']}: {r['tie_rate']:.1%} tie rate, {r['win_rate']:.1%} win rate")
print("\n   High tie rates often signal that these models are close in quality to their opponents,")
print("   OR that their responses are generic enough that humans can't differentiate.")


💡 Top 5 models with highest tie rates:
   qwen1.5-4b-chat: 38.0% tie rate, 17.5% win rate
   openhermes-2.5-mistral-7b: 36.0% tie rate, 29.4% win rate
   dolphin-2.2.1-mistral-7b: 35.7% tie rate, 25.5% win rate
   zephyr-7b-alpha: 35.2% tie rate, 28.9% win rate
   openchat-3.5-0106: 34.8% tie rate, 25.8% win rate

   High tie rates often signal that these models are close in quality to their opponents,
   OR that their responses are generic enough that humans can't differentiate.


In [9]:
# --- Which MODEL PAIRS produce the most ties? ---
df['unordered_pair'] = df.apply(
    lambda r: ' vs '.join(sorted([r['model_a'], r['model_b']])), axis=1
)

pair_stats = df.groupby('unordered_pair').agg(
    total=('winner_tie', 'count'),
    ties=('winner_tie', 'sum')
).reset_index()
pair_stats['tie_rate'] = pair_stats['ties'] / pair_stats['total']
pair_stats = pair_stats[pair_stats['total'] >= 30]

top_tie_pairs = pair_stats.nlargest(15, 'tie_rate').sort_values('tie_rate')

fig = go.Figure(go.Bar(
    y=top_tie_pairs['unordered_pair'],
    x=top_tie_pairs['tie_rate'],
    orientation='h',
    marker=dict(
        color=top_tie_pairs['tie_rate'],
        colorscale='Teal',
        showscale=True
    ),
    customdata=top_tie_pairs['total'],
    hovertemplate='<b>%{y}</b><br>Tie Rate: %{x:.1%}<br>Battles: %{customdata}<extra></extra>'
))

fig.update_layout(
    title=dict(text='Top 15 Model Pairs with Highest Tie Rates (min. 30 battles)', font_size=20),
    xaxis=dict(title='Tie Rate', tickformat='.0%'),
    yaxis=dict(title=''),
    height=520
)
fig.show()

print("\n💡 Same-family matchups (e.g., gpt-4 vs gpt-4-turbo) tend to tie more —")
print("   humans struggle to differentiate models that are closely related.")


💡 Same-family matchups (e.g., gpt-4 vs gpt-4-turbo) tend to tie more —
   humans struggle to differentiate models that are closely related.


## Chapter 6: Head-to-Head Dominance Matrix
Which models beat which? This heatmap reveals the hidden dominance hierarchy.

In [10]:
# Select top N models by battle count for a readable matrix
TOP_N = 15
top_models = model_stats.nlargest(TOP_N, 'total')['model'].tolist()

matrix = pd.DataFrame(np.nan, index=top_models, columns=top_models)

for _, row in df.iterrows():
    a, b = row['model_a'], row['model_b']
    if a not in top_models or b not in top_models:
        continue
    # Record: row = winner, col = loser (win rate of row model vs col model)
    if pd.isna(matrix.loc[a, b]):
        matrix.loc[a, b] = 0
        matrix.loc[b, a] = 0
    if row['winner_model_a'] == 1:
        matrix.loc[a, b] = matrix.loc[a, b] + 1 if not pd.isna(matrix.loc[a, b]) else 1
    elif row['winner_model_b'] == 1:
        matrix.loc[b, a] = matrix.loc[b, a] + 1 if not pd.isna(matrix.loc[b, a]) else 1

# Convert wins to win rates
counts = pd.DataFrame(0, index=top_models, columns=top_models)
for _, row in df.iterrows():
    a, b = row['model_a'], row['model_b']
    if a in top_models and b in top_models:
        counts.loc[a, b] += 1
        counts.loc[b, a] += 1

win_rate_matrix = (matrix / counts.replace(0, np.nan)).round(2)

fig = go.Figure(go.Heatmap(
    z=win_rate_matrix.values,
    x=win_rate_matrix.columns.tolist(),
    y=win_rate_matrix.index.tolist(),
    colorscale='RdYlGn',
    zmid=0.5,
    zmin=0, zmax=1,
    hovertemplate='<b>%{y}</b> vs <b>%{x}</b><br>Win Rate: %{z:.0%}<extra></extra>',
    colorbar=dict(title='Win Rate')
))

fig.update_layout(
    title=dict(text=f'Head-to-Head Win Rate Matrix (Top {TOP_N} Models)', font_size=20),
    xaxis=dict(title='Opponent (column model)', tickangle=-40),
    yaxis=dict(title='Model (row model)'),
    height=580
)
fig.show()

print("\n💡 Green = row model wins more against column model. Red = row model loses more.")
print("   GPT-4 variants show consistent green rows — true arena dominants.")


💡 Green = row model wins more against column model. Red = row model loses more.
   GPT-4 variants show consistent green rows — true arena dominants.


## Chapter 7: The Verdict — Final Insights
A visual summary of everything we learned.

In [11]:
# --- Ranking: Win Rate vs Tie Rate vs Popularity ---
top20 = reliable.nlargest(20, 'total').copy()
top20['loss_rate'] = 1 - top20['win_rate'] - top20['tie_rate']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Win Rate by Model (top 20 by battles)', 'Battles per Model (popularity)']
)

sorted_wr = top20.sort_values('win_rate', ascending=True)
fig.add_trace(go.Bar(
    y=sorted_wr['model'], x=sorted_wr['win_rate'],
    orientation='h', marker_color=COLORS['model_a'],
    name='Win Rate',
    hovertemplate='<b>%{y}</b><br>Win Rate: %{x:.1%}<extra></extra>'
), row=1, col=1)

sorted_total = top20.sort_values('total', ascending=True)
fig.add_trace(go.Bar(
    y=sorted_total['model'], x=sorted_total['total'],
    orientation='h', marker_color=COLORS['accent'],
    name='Total Battles',
    hovertemplate='<b>%{y}</b><br>Total Battles: %{x:,}<extra></extra>'
), row=1, col=2)

fig.update_layout(
    title=dict(text='Final Summary: Who Dominates the Arena?', font_size=22),
    height=600, showlegend=False
)
fig.update_xaxes(tickformat='.0%', row=1, col=1)
fig.show()

## Key Takeaways & Dashboard Direction

---

### 📖 The Story in Three Sentences
The LMSYS arena is a fair, well-balanced battleground where GPT-4 variants consistently dominate — winning over 50% of their battles regardless of position. Longer responses have a modest positive effect on winning, but the identity of the model matters far more than response length. Ties cluster among similarly-matched or similar-family models, suggesting humans struggle to differentiate peers.

---

### 🔑 Key Insights

| # | Insight | Evidence |
|---|---------|----------|
| 1 | Arena is balanced — no position bias at aggregate level | A/B win rates ≈ 35% each |
| 2 | GPT-4 variants are true dominants | Consistently highest win rates across positions |
| 3 | Longer responses win more, but weakly | Moderate correlation; being *much* shorter is penalized |
| 4 | Ties reveal model similarity | Same-family pairs tie most frequently |
| 5 | Some models have position sensitivity | Scatter plot shows outliers off the diagonal |

---

### 🚀 Final Project Dashboard Plan

**Title:** *The AI Arena: A Live Chatbot Leaderboard*

**Dashboard Panels:**
1. **Leaderboard** — Ranked model cards with win/tie/loss rates
2. **Head-to-Head Explorer** — Pick any two models, see their battle history
3. **Length vs Win Rate** — Interactive scatter with filters
4. **Tie Analysis** — Which matchups can't humans decide?
5. **Position Bias Check** — Model-level A vs B performance comparison

**Target Audience:** AI researchers and enthusiasts curious about which models perform best in blind human evaluations.